# Slack and Discord alerts for new SEC filings

Post new filings that match your queries to a Slack or Discord channel with an incoming webhook and a short Python script.

Read it with charts at https://www.quantisnow.com/sec-api/guides/slack-discord-filing-alerts. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Both Slack and Discord take a message on an incoming webhook URL. Poll the API for new matches, remember which ids you've posted, and send the rest.

## The script

In [ ]:
import json, os, pathlib, requests

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]
HOOK = os.environ["WEBHOOK_URL"]          # a Slack or Discord incoming webhook
SEEN = pathlib.Path("seen.json")
seen = set(json.loads(SEEN.read_text())) if SEEN.exists() else set()

QUERIES = {
    "8-K for our names": ("filings", 'ticker:(AAPL OR MSFT OR NVDA) AND formType:"8-K" AND filedAt:>=2026-11-23'),
    "Insider buys over $1M": ("insider-trades", "formType:4 AND transactionCodes:P AND purchased.valueUsd:>1000000 AND filedAt:>=2026-11-23"),
}

for label, (path, q) in QUERIES.items():
    rows = requests.post(f"{API}/{path}", headers={"Authorization": f"Bearer {KEY}"},
                         json={"query": q, "size": 50, "sort": "filedAt:desc"}).json()["data"]
    for r in reversed(rows):
        if r["id"] in seen:
            continue
        seen.add(r["id"])
        name = r.get("companyName") or r.get("issuer", {}).get("name")
        text = f"{label}: {name} {r.get('formType', '')} {r['linkToFilingIndex']}"
        # Slack reads "text", Discord reads "content"; sending both works for either
        requests.post(HOOK, json={"text": text, "content": text})

SEEN.write_text(json.dumps(sorted(seen)))

Run it every few minutes with cron. Replace the date in each query with today's (`date +%F`) so it only looks at today's filings, and keep `seen.json` between runs. Two queries every five minutes during market hours is about 3,400 calls a month.

Related: [Daily screener with cron](https://www.quantisnow.com/sec-api/guides/daily-sec-screener-cron), [Real-time filings](https://www.quantisnow.com/sec-api/guides/real-time-sec-filings-api).